In [0]:
%pip install databricks-ai-search
dbutils.library.restartPython()

In [0]:
from databricks.ai_search.client import AISearchClient

client = AISearchClient(disable_notice=True)
index = client.get_index(
    endpoint_name="privacylens-endpoint",
    index_name="workspace.privacylens.silver_chunks_index",
)

COLUMNS = ["chunk_id", "law", "section_num", "citation", "page_start", "chunk_text"]

def retrieve(question, law=None, k=5, query_type="HYBRID"):
    kwargs = dict(query_text=question, columns=COLUMNS,
                  num_results=k, query_type=query_type)
    if law:
        kwargs["filters"] = {"law": law}
    res = index.similarity_search(**kwargs)
    cols = [c["name"] for c in res["manifest"]["columns"]]
    return [dict(zip(cols, row)) for row in res["result"].get("data_array", [])]

def retrieve_balanced(question, k_per_law=3, query_type="HYBRID"):
    return (retrieve(question, "GDPR", k_per_law, query_type)
            + retrieve(question, "DPDP", k_per_law, query_type))

In [0]:
def score(fn):
    found_total, misses = 0, []
    for q, expected in tests:
        got = {f"{h['law']}_{h['section_num']}" for h in fn(q)}
        found_total += len(expected & got)
        misses += sorted(expected - got)
    return found_total, misses

configs = {
    "HYBRID, top 6":     lambda q: retrieve(q, k=6, query_type="HYBRID"),
    "ANN, top 6":        lambda q: retrieve(q, k=6, query_type="ANN"),
    "HYBRID, 3 per law": lambda q: retrieve_balanced(q, 3, "HYBRID"),
    "ANN, 3 per law":    lambda q: retrieve_balanced(q, 3, "ANN"),
}

for name, fn in configs.items():
    found, misses = score(fn)
    print(f"{name:18} {found}/10 found | missed: {misses}")

In [0]:
def retrieve_context(question, law=None):
    """PrivacyLens retrieval: ANN search.
    Both laws by default (3 cards each); 5 cards when one law is chosen."""
    if law:
        return retrieve(question, law=law, k=5, query_type="ANN")
    return retrieve_balanced(question, k_per_law=3, query_type="ANN")

for h in retrieve_context("Can a user make a company delete their data?"):
    print(h["citation"], "| page", int(h["page_start"]))